# Turbofan Engine Remaining Useful Life (RUL) Prediction
### NASA C-MAPSS Dataset — Machine Learning & Deep Learning Pipeline

This notebook predicts the **Remaining Useful Life (RUL)** of turbofan engines from multivariate sensor time series.

**Workflow**
1. Data Loading
2. Data Preprocessing
3. Data Visualization
4. Machine Learning Model Implementation
5. Machine Learning Model Evaluation
6. Deep Learning Model Implementation
7. Deep Learning Model Evaluation
8. Final Comparison & Test Predictions

## 0. Setup & Imports

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print('TensorFlow version :', tf.__version__)
print('GPU available      :', len(tf.config.list_physical_devices('GPU')) > 0)
print('XGBoost available  :', XGB_AVAILABLE)

TensorFlow version : 2.21.0
GPU available      : False
XGBoost available  : True


### Global Configuration

In [ ]:
MAX_RUL       = 125    # piece-wise linear RUL cap (engines degrade little early in life)
VAL_SIZE      = 0.20   # fraction of training ENGINES held out for validation
ROLL_WINDOW   = 10     # rolling window used for ML features
SEQ_LEN       = 30     # sequence length for deep learning models
BATCH_SIZE    = 256
EPOCHS        = 60

## 1. Data Loading

The competition files are downloaded with `kagglehub`. If the `train` and `test` DataFrames are already in memory,
they are used directly; otherwise they are read from the downloaded files. The training RUL is computed from each
engine's last cycle when it is not already present.

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('ml-olympiad-predictive-maintenance')

print("Path to competition files:", path)

In [ ]:
TRAIN_FILE = os.path.join(path, 'train_set.csv')
TEST_FILE  = os.path.join(path, 'test_set.csv')

COLUMNS = ['unit_nr', 'time_cycles', 'setting_1', 'setting_2', 'setting_3',
           'Fan inlet temperature (◦R)', 'LPC outlet temperature (◦R)',
           'HPC outlet temperature (◦R)', 'LPT outlet temperature (◦R)',
           'Fan inlet Pressure (psia)', 'bypass-duct pressure (psia)',
           'HPC outlet pressure (psia)', 'Physical fan speed (rpm)',
           'Physical core speed (rpm)', 'Engine pressure ratio (P50/P2)',
           'HPC outlet Static pressure (psia)',
           'Ratio of fuel flow to Ps30 (pps/psia)', 'Corrected fan speed (rpm)',
           'Corrected core speed (rpm)', 'Bypass Ratio', 'Burner fuel-air ratio',
           'Bleed Enthalpy', 'Required fan speed', 'Required fan conversion speed',
           'High-pressure turbines Cool air flow',
           'Low-pressure turbines Cool air flow']

def load_raw(path):
    df = pd.read_csv(path)
    df = df.loc[:, ~df.columns.astype(str).str.startswith('Unnamed')]
    extra_cols = list(df.columns[len(COLUMNS):])
    df.columns = COLUMNS + extra_cols
    df = df.apply(pd.to_numeric, errors='coerce')
    df = df.dropna(subset=['unit_nr', 'time_cycles'])
    df['unit_nr'] = df['unit_nr'].astype(int)
    df['time_cycles'] = df['time_cycles'].astype(int)
    return df

def add_train_rul(df):
    max_cycle = df.groupby('unit_nr')['time_cycles'].transform('max')
    df['RUL'] = max_cycle - df['time_cycles']
    return df

train = load_raw(TRAIN_FILE)
test  = load_raw(TEST_FILE)
if 'RUL' not in train.columns:
    train = add_train_rul(train)

train_df = train.copy().sort_values(['unit_nr', 'time_cycles']).reset_index(drop=True)
test_df  = test.copy().sort_values(['unit_nr', 'time_cycles']).reset_index(drop=True)

print('Train shape:', train_df.shape, '| engines:', train_df['unit_nr'].nunique())
print('Test shape :', test_df.shape,  '| engines:', test_df['unit_nr'].nunique())

### 1.1 First Look at the Data

In [ ]:
display(train_df.head())
display(test_df.head())

In [ ]:
train_df.info()

In [ ]:
display(train_df.describe().T)

## 2. Data Preprocessing

### 2.1 Missing Values & Duplicates

In [ ]:
print('Missing values (train):', train_df.isnull().sum().sum())
print('Missing values (test) :', test_df.isnull().sum().sum())
print('Duplicate rows (train):', train_df.duplicated().sum())
print('Duplicate rows (test) :', test_df.duplicated().sum())

train_df = train_df.drop_duplicates().reset_index(drop=True)
test_df  = test_df.drop_duplicates().reset_index(drop=True)

### 2.2 Feature Groups

In [ ]:
ID_COLS      = ['unit_nr', 'time_cycles']
SETTING_COLS = ['setting_1', 'setting_2', 'setting_3']
SENSOR_COLS  = [c for c in train_df.columns if c not in ID_COLS + SETTING_COLS + ['RUL']]

# Cast sensors to float so normalised values can be written back
train_df[SENSOR_COLS] = train_df[SENSOR_COLS].astype(float)
test_df[SENSOR_COLS]  = test_df[SENSOR_COLS].astype(float)

print(f'{len(SETTING_COLS)} operational settings, {len(SENSOR_COLS)} sensors')

### 2.3 Operating Condition Identification

The three operational settings form discrete flight regimes. Sensor readings depend strongly on the regime,
which hides the degradation signal. We identify the regimes with K-Means clustering on the settings
(the number of clusters is taken from the number of distinct rounded setting combinations).

In [ ]:
rounded = train_df[SETTING_COLS].copy()
rounded['setting_1'] = rounded['setting_1'].round(0) + 0.0
rounded['setting_2'] = rounded['setting_2'].round(2) + 0.0
rounded['setting_3'] = rounded['setting_3'].round(0) + 0.0
n_unique = rounded.drop_duplicates().shape[0]
N_CONDITIONS = n_unique if n_unique <= 10 else 6
print('Number of operating conditions:', N_CONDITIONS)

setting_scaler = StandardScaler().fit(train_df[SETTING_COLS])
kmeans = KMeans(n_clusters=N_CONDITIONS, n_init=10, random_state=SEED)
train_df['op_condition'] = kmeans.fit_predict(setting_scaler.transform(train_df[SETTING_COLS]))
test_df['op_condition']  = kmeans.predict(setting_scaler.transform(test_df[SETTING_COLS]))

display(train_df.groupby('op_condition')[SETTING_COLS].agg(['mean', 'count']).round(3))

### 2.4 Condition-Based Normalization

Each sensor is standardised **within its operating condition** using statistics computed on the training set only
(prevents leakage). This removes the regime effect and leaves the degradation trend.

In [ ]:
cond_mean = train_df.groupby('op_condition')[SENSOR_COLS].mean()
cond_std  = train_df.groupby('op_condition')[SENSOR_COLS].std().replace(0, 1).fillna(1)

def condition_normalize(df):
    out = df.copy()
    for cond in out['op_condition'].unique():
        idx = out['op_condition'] == cond
        out.loc[idx, SENSOR_COLS] = (out.loc[idx, SENSOR_COLS] - cond_mean.loc[cond]) / cond_std.loc[cond]
    return out

train_norm = condition_normalize(train_df)
test_norm  = condition_normalize(test_df)

### 2.5 Remove Non-Informative Sensors

In [ ]:
# Sensors that are constant after normalisation carry no degradation information
sensor_std = train_norm[SENSOR_COLS].std()
DROP_SENSORS = sensor_std[sensor_std < 1e-3].index.tolist()

# Sensors with almost no correlation to RUL are also dropped
rul_corr = train_norm[SENSOR_COLS].corrwith(train_norm['RUL']).abs()
DROP_SENSORS += [c for c in rul_corr[rul_corr < 0.01].index if c not in DROP_SENSORS]

FEATURES = [c for c in SENSOR_COLS if c not in DROP_SENSORS]
print('Dropped sensors:', DROP_SENSORS)
print(f'Kept {len(FEATURES)} sensors:', FEATURES)

### 2.6 Target Engineering — Piece-wise Linear RUL

Early in life an engine shows no measurable degradation, so a constant RUL is assumed until degradation begins.
The target is capped at `MAX_RUL`.

In [ ]:
train_norm['RUL_clipped'] = train_norm['RUL'].clip(upper=MAX_RUL)
display(train_norm[['unit_nr', 'time_cycles', 'RUL', 'RUL_clipped']].head())

### 2.7 Feature Engineering for Machine Learning

Tabular ML models see one row at a time, so temporal context is added through rolling mean and rolling standard
deviation of each sensor (computed per engine, using only past cycles).

In [ ]:
def add_rolling_features(df, cols, window=ROLL_WINDOW):
    df = df.sort_values(['unit_nr', 'time_cycles']).copy()
    grp = df.groupby('unit_nr')[cols]
    r_mean = grp.transform(lambda s: s.rolling(window, min_periods=1).mean())
    r_std  = grp.transform(lambda s: s.rolling(window, min_periods=1).std()).fillna(0)
    r_mean.columns = [f'{c}_rmean' for c in cols]
    r_std.columns  = [f'{c}_rstd' for c in cols]
    return pd.concat([df, r_mean, r_std], axis=1)

train_fe = add_rolling_features(train_norm, FEATURES)
test_fe  = add_rolling_features(test_norm,  FEATURES)

ML_FEATURES = (['time_cycles'] + FEATURES
               + [f'{c}_rmean' for c in FEATURES]
               + [f'{c}_rstd'  for c in FEATURES])
print('Number of ML features:', len(ML_FEATURES))

### 2.8 Train / Validation Split (by Engine)

The split is performed on **engine IDs**, not rows, so no engine appears in both sets (avoids temporal leakage).

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=SEED)
tr_idx, val_idx = next(gss.split(train_fe, groups=train_fe['unit_nr']))
train_units = train_fe.iloc[tr_idx]['unit_nr'].unique()
val_units   = train_fe.iloc[val_idx]['unit_nr'].unique()

tr_set  = train_fe[train_fe['unit_nr'].isin(train_units)].reset_index(drop=True)
val_set = train_fe[train_fe['unit_nr'].isin(val_units)].reset_index(drop=True)

X_train, y_train = tr_set[ML_FEATURES].values,  tr_set['RUL_clipped'].values
X_val,   y_val   = val_set[ML_FEATURES].values, val_set['RUL_clipped'].values

# Last cycle of every validation engine (standard C-MAPSS evaluation point)
val_last_mask = val_set.groupby('unit_nr')['time_cycles'].transform('max') == val_set['time_cycles']

# Last cycle of every test engine (for final predictions)
test_last = test_fe.groupby('unit_nr').tail(1).reset_index(drop=True)
X_test_last = test_last[ML_FEATURES].values

print(f'Train engines: {len(train_units)} | rows: {len(tr_set)}')
print(f'Val engines  : {len(val_units)} | rows: {len(val_set)}')
print(f'Test engines : {len(test_last)}')

## 3. Data Visualization

### 3.1 Engine Lifetime Distribution

In [ ]:
life = train_df.groupby('unit_nr')['time_cycles'].max()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(life, bins=30, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Distribution of Engine Lifetimes (Train)')
axes[0].set_xlabel('Total cycles until failure')

test_len = test_df.groupby('unit_nr')['time_cycles'].max()
sns.histplot(test_len, bins=30, kde=True, ax=axes[1], color='darkorange')
axes[1].set_title('Distribution of Observed Cycles (Test)')
axes[1].set_xlabel('Cycles observed')
plt.tight_layout(); plt.show()

print(f'Train lifetime -> min: {life.min()}, max: {life.max()}, mean: {life.mean():.1f}')

### 3.2 RUL Distribution (Raw vs Clipped)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(train_norm['RUL'], bins=50, ax=axes[0], color='teal')
axes[0].set_title('Raw RUL')
sns.histplot(train_norm['RUL_clipped'], bins=50, ax=axes[1], color='crimson')
axes[1].set_title(f'Clipped RUL (cap = {MAX_RUL})')
plt.tight_layout(); plt.show()

### 3.3 Operating Conditions

In [ ]:
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
sample = train_df.sample(min(5000, len(train_df)), random_state=SEED)
sc = ax.scatter(sample['setting_1'], sample['setting_2'], sample['setting_3'],
                c=sample['op_condition'], cmap='tab10', s=8)
ax.set_xlabel('setting_1'); ax.set_ylabel('setting_2'); ax.set_zlabel('setting_3')
ax.set_title('Operating Conditions Identified by K-Means')
plt.colorbar(sc, label='Condition', shrink=0.6)
plt.show()

sns.countplot(x='op_condition', data=train_df, palette='tab10')
plt.title('Rows per Operating Condition'); plt.show()

### 3.4 Sensor Signal: Raw vs Condition-Normalized

In [ ]:
example_sensor = FEATURES[0]
unit_id = train_df['unit_nr'].iloc[0]
raw_u  = train_df[train_df['unit_nr'] == unit_id]
norm_u = train_norm[train_norm['unit_nr'] == unit_id]

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
axes[0].plot(raw_u['time_cycles'], raw_u[example_sensor], lw=0.8)
axes[0].set_title(f'Raw — {example_sensor} (engine {unit_id})')
axes[1].plot(norm_u['time_cycles'], norm_u[example_sensor], lw=0.8, color='green')
axes[1].plot(norm_u['time_cycles'], norm_u[example_sensor].rolling(10, min_periods=1).mean(), color='black', lw=2, label='rolling mean')
axes[1].set_title(f'Condition-normalized — {example_sensor}')
axes[1].legend()
for a in axes: a.set_xlabel('Cycle')
plt.tight_layout(); plt.show()

### 3.5 Sensor Degradation Trends vs RUL

In [ ]:
sample_units = train_norm['unit_nr'].unique()[:8]
n_cols = 3
n_rows = int(np.ceil(len(FEATURES) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 3.5 * n_rows))
axes = axes.flatten()
for i, col in enumerate(FEATURES):
    for u in sample_units:
        d = train_norm[train_norm['unit_nr'] == u]
        axes[i].plot(d['RUL'], d[col].rolling(10, min_periods=1).mean(), lw=1)
    axes[i].invert_xaxis()
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('RUL (cycles to failure)')
for j in range(i + 1, len(axes)):
    axes[j].axis('off')
plt.suptitle('Smoothed Normalized Sensor Trends Approaching Failure (8 engines)', y=1.01, fontsize=14)
plt.tight_layout(); plt.show()

### 3.6 Correlation Analysis

In [ ]:
corr = train_norm[FEATURES + ['RUL']].corr()
plt.figure(figsize=(14, 11))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=True, fmt='.2f', annot_kws={'size': 7})
plt.title('Correlation Matrix (Normalized Sensors & RUL)')
plt.show()

In [ ]:
rul_corr_sorted = corr['RUL'].drop('RUL').sort_values()
plt.figure(figsize=(10, 6))
rul_corr_sorted.plot(kind='barh', color=np.where(rul_corr_sorted > 0, 'seagreen', 'indianred'))
plt.title('Correlation of Each Sensor with RUL')
plt.xlabel('Pearson correlation')
plt.show()

### 3.7 Sensor Distributions

In [ ]:
n_rows = int(np.ceil(len(FEATURES) / 4))
fig, axes = plt.subplots(n_rows, 4, figsize=(18, 3 * n_rows))
axes = axes.flatten()
for i, col in enumerate(FEATURES):
    sns.kdeplot(train_norm[col], ax=axes[i], fill=True, color='slateblue')
    axes[i].set_title(col, fontsize=9); axes[i].set_xlabel('')
for j in range(i + 1, len(axes)):
    axes[j].axis('off')
plt.tight_layout(); plt.show()

## 4. Machine Learning Model Implementation

### 4.1 Evaluation Utilities

Metrics used:
- **RMSE** — primary metric, penalises large errors
- **MAE** — average absolute error in cycles
- **R²** — explained variance
- **NASA Score** — asymmetric scoring function from the PHM08 challenge; late predictions (overestimating RUL) are penalised more heavily than early ones. Lower is better.

In [ ]:
def nasa_score(y_true, y_pred):
    d = np.asarray(y_pred) - np.asarray(y_true)
    return float(np.sum(np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)))

def evaluate(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return {
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE' : mean_absolute_error(y_true, y_pred),
        'R2'  : r2_score(y_true, y_pred),
        'NASA_Score': nasa_score(y_true, y_pred),
    }

results = []          # collects metrics of every model
predictions = {}      # stores validation / test predictions for plotting

### 4.2 Model Definitions

| Model | Why |
|---|---|
| Linear Regression | Simple, interpretable baseline |
| Random Forest | Robust non-linear bagging ensemble |
| Histogram Gradient Boosting | Fast, strong boosting model |
| XGBoost (fallback: Extra Trees) | State-of-the-art gradient boosting |

In [ ]:
ml_models = {
    'Linear Regression': Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())]),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_leaf=5,
                                           max_features=0.5, n_jobs=-1, random_state=SEED),
    'Hist Gradient Boosting': HistGradientBoostingRegressor(max_iter=500, learning_rate=0.05,
                                                            max_leaf_nodes=31, l2_regularization=1.0,
                                                            early_stopping=True, random_state=SEED),
}
if XGB_AVAILABLE:
    ml_models['XGBoost'] = XGBRegressor(n_estimators=600, learning_rate=0.03, max_depth=6,
                                        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                                        n_jobs=-1, random_state=SEED)
else:
    ml_models['Extra Trees'] = ExtraTreesRegressor(n_estimators=300, max_depth=18, min_samples_leaf=3,
                                                   n_jobs=-1, random_state=SEED)
list(ml_models.keys())

### 4.3 Model Training

In [ ]:
import time
trained_ml = {}
for name, model in ml_models.items():
    start = time.time()
    model.fit(X_train, y_train)
    trained_ml[name] = model
    print(f'{name:<25} trained in {time.time() - start:6.1f}s')

## 5. Machine Learning Model Evaluation

### 5.1 Metrics on Validation Engines

Two views are reported:
- **All cycles** — accuracy over each engine's full history
- **Last cycle** — prediction at the final observed cycle of each engine (the standard C-MAPSS protocol, mirroring the test set)

In [ ]:
y_val_last = y_val[val_last_mask.values]

for name, model in trained_ml.items():
    pred_all  = np.clip(model.predict(X_val), 0, MAX_RUL)
    pred_last = pred_all[val_last_mask.values]
    predictions[name] = {'val_all': pred_all, 'val_last': pred_last,
                         'test': np.clip(model.predict(X_test_last), 0, MAX_RUL)}

    m_all, m_last = evaluate(y_val, pred_all), evaluate(y_val_last, pred_last)
    row = {'Model': name, 'Type': 'ML',
           'RMSE (all)': m_all['RMSE'], 'MAE (all)': m_all['MAE'], 'R2 (all)': m_all['R2'],
           'RMSE (last)': m_last['RMSE'], 'MAE (last)': m_last['MAE'], 'R2 (last)': m_last['R2'],
           'NASA Score (last)': m_last['NASA_Score']}
    results.append(row)

ml_results = pd.DataFrame([r for r in results if r['Type'] == 'ML']).set_index('Model')
display(ml_results.round(3))

### 5.2 Model Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric, color in zip(axes, ['RMSE (last)', 'MAE (last)', 'R2 (last)'], ['#4C72B0', '#DD8452', '#55A868']):
    ml_results[metric].sort_values().plot(kind='barh', ax=ax, color=color)
    ax.set_title(metric)
    for i, v in enumerate(ml_results[metric].sort_values()):
        ax.text(v, i, f' {v:.2f}', va='center')
plt.suptitle('ML Models — Validation (last cycle per engine)', fontsize=14)
plt.tight_layout(); plt.show()

### 5.3 Predicted vs Actual RUL

In [ ]:
fig, axes = plt.subplots(1, len(trained_ml), figsize=(5 * len(trained_ml), 5))
for ax, name in zip(np.atleast_1d(axes), trained_ml):
    ax.scatter(y_val, predictions[name]['val_all'], s=3, alpha=0.3)
    ax.plot([0, MAX_RUL], [0, MAX_RUL], 'r--')
    ax.set_title(name); ax.set_xlabel('Actual RUL'); ax.set_ylabel('Predicted RUL')
plt.tight_layout(); plt.show()

### 5.4 Residual Analysis

In [ ]:
fig, axes = plt.subplots(1, len(trained_ml), figsize=(5 * len(trained_ml), 4))
for ax, name in zip(np.atleast_1d(axes), trained_ml):
    residuals = predictions[name]['val_all'] - y_val
    sns.histplot(residuals, bins=50, kde=True, ax=ax, color='purple')
    ax.axvline(0, color='red', ls='--')
    ax.set_title(f'{name}\nmean={residuals.mean():.2f}, std={residuals.std():.2f}')
    ax.set_xlabel('Predicted - Actual')
plt.tight_layout(); plt.show()

### 5.5 Feature Importance (Best Tree Model)

In [ ]:
tree_models = {k: v for k, v in trained_ml.items() if hasattr(v, 'feature_importances_')}
best_tree = min(tree_models, key=lambda k: ml_results.loc[k, 'RMSE (last)'])
importances = pd.Series(tree_models[best_tree].feature_importances_, index=ML_FEATURES).sort_values(ascending=False)

plt.figure(figsize=(10, 8))
importances.head(20)[::-1].plot(kind='barh', color='darkcyan')
plt.title(f'Top 20 Feature Importances — {best_tree}')
plt.tight_layout(); plt.show()

### 5.6 RUL Trajectories for Sample Validation Engines

In [ ]:
best_ml = ml_results['RMSE (last)'].idxmin()
print('Best ML model:', best_ml)

sample_val_units = val_units[:4]
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, u in zip(axes, sample_val_units):
    mask = (val_set['unit_nr'] == u).values
    ax.plot(val_set.loc[mask, 'time_cycles'], y_val[mask], 'k-', label='True (clipped)')
    ax.plot(val_set.loc[mask, 'time_cycles'], predictions[best_ml]['val_all'][mask], 'b-', alpha=0.7, label='Predicted')
    ax.set_title(f'Engine {u}'); ax.set_xlabel('Cycle'); ax.set_ylabel('RUL')
    ax.legend()
plt.suptitle(f'{best_ml} — RUL over engine life', fontsize=14)
plt.tight_layout(); plt.show()

## 6. Deep Learning Model Implementation

### 6.1 Sequence Generation

Deep learning models consume **sliding windows** of `SEQ_LEN` consecutive cycles. Each engine's history is padded
at the start (repeating its first reading) so every cycle — including engines shorter than `SEQ_LEN` — produces a window.
The target is scaled to `[0, 1]` by dividing by `MAX_RUL` for stable training.

In [ ]:
DL_FEATURES = FEATURES

def make_sequences(df, features, seq_len=SEQ_LEN, target_col=None, last_only=False):
    X, y = [], []
    for _, g in df.groupby('unit_nr', sort=True):
        arr = g[features].values.astype('float32')
        padded = np.vstack([np.repeat(arr[:1], seq_len - 1, axis=0), arr])
        windows = np.lib.stride_tricks.sliding_window_view(padded, (seq_len, arr.shape[1]))[:, 0]
        if last_only:
            windows = windows[-1:]
        X.append(windows)
        if target_col is not None:
            t = g[target_col].values.astype('float32')
            y.append(t[-1:] if last_only else t)
    X = np.concatenate(X)
    return (X, np.concatenate(y)) if target_col is not None else X

tr_norm  = train_norm[train_norm['unit_nr'].isin(train_units)].sort_values(['unit_nr', 'time_cycles'])
val_norm = train_norm[train_norm['unit_nr'].isin(val_units)].sort_values(['unit_nr', 'time_cycles'])

X_tr_seq,   y_tr_seq   = make_sequences(tr_norm,  DL_FEATURES, target_col='RUL_clipped')
X_val_seq,  y_val_seq  = make_sequences(val_norm, DL_FEATURES, target_col='RUL_clipped')
X_val_last, y_val_last_seq = make_sequences(val_norm, DL_FEATURES, target_col='RUL_clipped', last_only=True)
X_test_seq = make_sequences(test_norm.sort_values(['unit_nr', 'time_cycles']), DL_FEATURES, last_only=True)

y_tr_scaled, y_val_scaled = y_tr_seq / MAX_RUL, y_val_seq / MAX_RUL

print('Train sequences :', X_tr_seq.shape)
print('Val sequences   :', X_val_seq.shape)
print('Val last-cycle  :', X_val_last.shape)
print('Test sequences  :', X_test_seq.shape)

### 6.2 Model Architectures

Three architectures are compared:
- **LSTM** — stacked Long Short-Term Memory network capturing long-term temporal dependencies
- **GRU** — lighter recurrent alternative to LSTM
- **1D-CNN** — convolutional network extracting local temporal patterns

In [ ]:
N_FEATURES = X_tr_seq.shape[2]

def build_lstm():
    inp = keras.Input(shape=(SEQ_LEN, N_FEATURES))
    x = layers.LSTM(64, return_sequences=True)(inp)
    x = layers.Dropout(0.2)(x)
    x = layers.LSTM(32)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(1)(x)
    return keras.Model(inp, out, name='LSTM')

def build_gru():
    inp = keras.Input(shape=(SEQ_LEN, N_FEATURES))
    x = layers.GRU(64, return_sequences=True)(inp)
    x = layers.Dropout(0.2)(x)
    x = layers.GRU(32)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(1)(x)
    return keras.Model(inp, out, name='GRU')

def build_cnn():
    inp = keras.Input(shape=(SEQ_LEN, N_FEATURES))
    x = layers.Conv1D(64, 5, padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(64, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(32, 3, padding='same', activation='relu')(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1)(x)
    return keras.Model(inp, out, name='CNN_1D')

dl_builders = {'LSTM': build_lstm, 'GRU': build_gru, '1D-CNN': build_cnn}
build_lstm().summary()

### 6.3 Model Training

Early stopping restores the best weights; the learning rate is reduced when validation loss plateaus.

In [ ]:
histories, trained_dl = {}, {}

for name, builder in dl_builders.items():
    print(f'\n========== Training {name} ==========')
    tf.keras.utils.set_random_seed(SEED)
    model = builder()
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss='mse', metrics=['mae'])
    cb = [
        callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5, verbose=1),
    ]
    history = model.fit(X_tr_seq, y_tr_scaled,
                        validation_data=(X_val_seq, y_val_scaled),
                        epochs=EPOCHS, batch_size=BATCH_SIZE,
                        callbacks=cb, verbose=2)
    histories[name], trained_dl[name] = history, model

## 7. Deep Learning Model Evaluation

### 7.1 Training Curves

In [ ]:
fig, axes = plt.subplots(1, len(histories), figsize=(6 * len(histories), 4))
for ax, (name, h) in zip(np.atleast_1d(axes), histories.items()):
    ax.plot(np.sqrt(h.history['loss']) * MAX_RUL, label='Train RMSE')
    ax.plot(np.sqrt(h.history['val_loss']) * MAX_RUL, label='Val RMSE')
    ax.set_title(f'{name} — Learning Curve'); ax.set_xlabel('Epoch'); ax.set_ylabel('RMSE (cycles)')
    ax.legend()
plt.tight_layout(); plt.show()

### 7.2 Metrics on Validation Engines

In [ ]:
for name, model in trained_dl.items():
    pred_all  = np.clip(model.predict(X_val_seq,  verbose=0).ravel() * MAX_RUL, 0, MAX_RUL)
    pred_last = np.clip(model.predict(X_val_last, verbose=0).ravel() * MAX_RUL, 0, MAX_RUL)
    pred_test = np.clip(model.predict(X_test_seq, verbose=0).ravel() * MAX_RUL, 0, MAX_RUL)
    predictions[name] = {'val_all': pred_all, 'val_last': pred_last, 'test': pred_test}

    m_all, m_last = evaluate(y_val_seq, pred_all), evaluate(y_val_last_seq, pred_last)
    row = {'Model': name, 'Type': 'DL',
           'RMSE (all)': m_all['RMSE'], 'MAE (all)': m_all['MAE'], 'R2 (all)': m_all['R2'],
           'RMSE (last)': m_last['RMSE'], 'MAE (last)': m_last['MAE'], 'R2 (last)': m_last['R2'],
           'NASA Score (last)': m_last['NASA_Score']}
    results.append(row)

dl_results = pd.DataFrame([r for r in results if r['Type'] == 'DL']).set_index('Model')
display(dl_results.round(3))

### 7.3 Predicted vs Actual RUL

In [ ]:
fig, axes = plt.subplots(1, len(trained_dl), figsize=(6 * len(trained_dl), 5))
for ax, name in zip(np.atleast_1d(axes), trained_dl):
    ax.scatter(y_val_seq, predictions[name]['val_all'], s=3, alpha=0.3, color='darkorange')
    ax.plot([0, MAX_RUL], [0, MAX_RUL], 'r--')
    ax.set_title(name); ax.set_xlabel('Actual RUL'); ax.set_ylabel('Predicted RUL')
plt.tight_layout(); plt.show()

### 7.4 Residual Analysis

In [ ]:
fig, axes = plt.subplots(1, len(trained_dl), figsize=(6 * len(trained_dl), 4))
for ax, name in zip(np.atleast_1d(axes), trained_dl):
    residuals = predictions[name]['val_all'] - y_val_seq
    sns.histplot(residuals, bins=50, kde=True, ax=ax, color='teal')
    ax.axvline(0, color='red', ls='--')
    ax.set_title(f'{name}\nmean={residuals.mean():.2f}, std={residuals.std():.2f}')
    ax.set_xlabel('Predicted - Actual')
plt.tight_layout(); plt.show()

### 7.5 RUL Trajectories for Sample Validation Engines

In [ ]:
best_dl = dl_results['RMSE (last)'].idxmin()
print('Best DL model:', best_dl)

val_norm_sorted = val_norm.reset_index(drop=True)
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, u in zip(axes, sample_val_units):
    mask = (val_norm_sorted['unit_nr'] == u).values
    ax.plot(val_norm_sorted.loc[mask, 'time_cycles'], y_val_seq[mask], 'k-', label='True (clipped)')
    ax.plot(val_norm_sorted.loc[mask, 'time_cycles'], predictions[best_dl]['val_all'][mask], 'g-', alpha=0.7, label=best_dl)
    ax.plot(val_norm_sorted.loc[mask, 'time_cycles'], predictions[best_ml]['val_all'][mask], 'b--', alpha=0.6, label=best_ml)
    ax.set_title(f'Engine {u}'); ax.set_xlabel('Cycle'); ax.set_ylabel('RUL')
    ax.legend(fontsize=8)
plt.suptitle('Best DL vs Best ML — RUL over engine life', fontsize=14)
plt.tight_layout(); plt.show()

## 8. Final Comparison — Machine Learning vs Deep Learning

In [ ]:
all_results = pd.DataFrame(results).set_index('Model').sort_values('RMSE (last)')
display(all_results.round(3))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
colors = ['#4C72B0' if t == 'ML' else '#DD8452' for t in all_results['Type']]
axes[0].barh(all_results.index, all_results['RMSE (last)'], color=colors)
axes[0].set_title('RMSE (last cycle, validation) — lower is better'); axes[0].invert_yaxis()
axes[1].barh(all_results.index, all_results['NASA Score (last)'], color=colors)
axes[1].set_title('NASA Score (last cycle, validation) — lower is better'); axes[1].invert_yaxis()
from matplotlib.patches import Patch
axes[1].legend(handles=[Patch(color='#4C72B0', label='ML'), Patch(color='#DD8452', label='DL')])
plt.tight_layout(); plt.show()

best_overall = all_results.index[0]
print(f'Best overall model: {best_overall}  (RMSE last-cycle = {all_results.loc[best_overall, "RMSE (last)"]:.2f})')

### 8.1 Error by True RUL Range (Best Model)

In [ ]:
y_ref = y_val if all_results.loc[best_overall, 'Type'] == 'ML' else y_val_seq
err_df = pd.DataFrame({'true': y_ref, 'abs_err': np.abs(predictions[best_overall]['val_all'] - y_ref)})
err_df['RUL bucket'] = pd.cut(err_df['true'], bins=[-1, 25, 50, 75, 100, MAX_RUL])
plt.figure(figsize=(10, 4))
sns.barplot(x='RUL bucket', y='abs_err', data=err_df, color='slateblue', errorbar=None)
plt.title(f'Mean Absolute Error by RUL Range — {best_overall}')
plt.ylabel('MAE (cycles)'); plt.show()

### 8.2 Test Set Predictions

The test set has no RUL labels, so all model scores above come from the held-out validation engines. Here each model predicts the RUL at the last observed cycle of every test engine.

In [ ]:
test_predictions = pd.DataFrame({'unit_nr': test_last['unit_nr'].values})
for name in all_results.index:
    test_predictions[f'RUL_{name}'] = np.round(predictions[name]['test'], 1)
test_predictions['RUL_Ensemble_Top3'] = test_predictions[[f'RUL_{m}' for m in all_results.index[:3]]].mean(axis=1).round(1)

display(test_predictions.head(10))
test_predictions.to_csv('test_rul_predictions.csv', index=False)
print('Saved: test_rul_predictions.csv')

In [ ]:
plt.figure(figsize=(10, 4))
sns.histplot(test_predictions[f'RUL_{best_overall}'], bins=30, color='seagreen')
plt.title(f'Distribution of Predicted Test RUL — {best_overall}'); plt.xlabel('Predicted RUL'); plt.show()

### 8.3 Save Best Models

In [ ]:
import joblib
os.makedirs('saved_models', exist_ok=True)
joblib.dump(trained_ml[best_ml], f"saved_models/{best_ml.replace(' ', '_')}.joblib")
trained_dl[best_dl].save(f"saved_models/{best_dl.replace('-', '_')}.keras")
print('Saved:', os.listdir('saved_models'))

## 9. Conclusion

- **Operating-condition normalization** was the key preprocessing step: it removes regime effects so sensor degradation trends become visible.
- **Piece-wise linear RUL (cap at 125)** makes the target learnable, since early-life cycles carry no degradation signal.
- **Rolling features** give tabular ML models temporal context; tree ensembles typically clearly beat the linear baseline.
- **Sequence models (LSTM / GRU / 1D-CNN)** learn temporal patterns directly from windows of raw normalized sensors and are usually the strongest single models.
- Prediction error is smallest near end of life (low RUL), which is exactly where maintenance decisions matter most.
- The NASA score penalizes late predictions, so a slightly conservative (under-estimating) model is preferable in practice.